# MLE for a Coin: the Bernoulli Case

Ten tosses, eight heads. Of three candidate coins ($p = 0.7, 0.5, 0.3$), which most likely produced the data? Compute each coin's probability of generating 8 heads and 2 tails, pick the largest — that's maximum likelihood. But why stop at three candidates? Let $p$ range over *all* coins: the likelihood $L(p) = p^8(1-p)^2$ becomes a **function of $p$** to maximize — take logs (products become sums), differentiate, set to zero, and out comes $\hat{p} = \frac{8}{10}$. In general, for $k$ heads in $n$ flips: $\hat{p} = \frac{k}{n}$ — the sample proportion, now *derived* rather than assumed. It is based on the third video of Lesson 6 (Point Estimation).

## 1. Introduction

The popcorn notebook chose among a handful of scenarios by comparing conditional probabilities. This notebook takes the decisive step for machine learning: the "scenarios" become a **continuum** of models — every coin bias $p \in [0, 1]$ — and picking the best one becomes a **calculus problem**. Along the way we meet the two workhorses of all ML training: the **likelihood function** and its friendlier twin, the **log-likelihood**.

After completing this notebook you will be able to:

- Compare finitely many models by their likelihoods (the three-coin contest).
- Write the likelihood of coin-flip data as a function of $p$ and maximize it with the **log trick** and a derivative.
- Derive the general Bernoulli MLE $\hat{p} = k/n$ and connect it to Lesson 5's sample proportion.
- Reproduce the derivation symbolically with sympy and numerically with grids.

## 2. Theory

### Three coins, one dataset

You toss a coin 10 times: **8 heads, 2 tails**. Three coins could have been used: coin 1 with $p = 0.7$, coin 2 fair ($p = 0.5$), coin 3 with $p = 0.3$. Which coin was it — or at least, which would you pick to regenerate the data? Compute each coin's probability of this outcome:

$$
P(8H2T \mid \text{coin 1}) = 0.7^8 \cdot 0.3^2 = 0.0051
$$
$$
P(8H2T \mid \text{coin 2}) = 0.5^{10} = 0.0010
$$
$$
P(8H2T \mid \text{coin 3}) = 0.3^8 \cdot 0.7^2 = 0.00003
$$

All small (any *specific* sequence is rare), but coin 1's is by far the largest: **pick coin 1** — maximum likelihood, exactly as with the popcorn.

### From three coins to all coins

Could we do better? Take a coin with heads-probability $p$ (and $1-p$ for tails). The probability that it generates 8 heads and 2 tails is

$$
L(p) = p^8 (1-p)^2
$$

This is the **likelihood** — the probability of seeing this data under the model "coin with parameter $p$" — and crucially it is a **function of $p$**. Finding the best coin means *maximizing* this function: a calculus problem.

### The log trick

Products of many small numbers are unpleasant (numerically and algebraically). The standard trick: take logarithms, turning products into sums:

$$
\log L(p) = 8 \log p + 2 \log(1-p)
$$

This is the **log-likelihood**. Since $\log$ is increasing, maximizing $\log L$ is the same as maximizing $L$ — and the log-likelihood is normally a much nicer function.

### Maximize: differentiate and set to zero

$$
\frac{d}{dp}\log L(p) = \frac{8}{p} - \frac{2}{1-p}
$$

(the $\frac{d}{dp}\log p = \frac1p$ rule, and a chain-rule minus sign for the second term). Setting this to zero: $8(1-p) = 2p \Rightarrow \hat{p} = \frac{8}{10}$.

The best possible coin has $p = 0.8$ — which makes sense: the data was 8 heads in 10 flips.

### The general case: $n$ flips, $k$ heads

Each flip $x_i$ is a Bernoulli($p$) variable, so its probability is $p^{x_i}(1-p)^{1-x_i}$ (giving $p$ when $x_i = 1$, $1-p$ when $x_i = 0$). The likelihood of the whole i.i.d. dataset is the product

$$
L(p) = \prod_{i=1}^{n} p^{x_i}(1-p)^{1-x_i} = p^{k}\,(1-p)^{n-k}
$$

with $k = \sum x_i$ the number of heads. Log-likelihood: $k \log p + (n-k)\log(1-p)$. Differentiate, set to zero:

$$
\frac{k}{\hat p} = \frac{n-k}{1-\hat p} \qquad\Longrightarrow\qquad \boxed{\hat{p} = \frac{k}{n}}
$$

The maximum-likelihood coin is the **sample mean of the flips** — the sample proportion of heads. Lesson 5 used $k/n$ because it was intuitive and unbiased; MLE *derives* it as the optimal answer to "which model most likely produced this data?".

## 3. Worked Examples

### Example 1: the three-coin contest, in full

$0.7^8 = 0.0576$, times $0.09$ → $0.00519$; $0.5^{10} = 0.000977$; $0.3^8 = 0.0000656$, times $0.49$ → $0.0000322$. Ratios: coin 1 beats coin 2 by 5.3×, and coin 3 by 161× — a decisive verdict from just 10 flips.

### Example 2: the derivative, slowly

$$
\frac{8}{p} - \frac{2}{1-p} = 0
\;\Rightarrow\; 8(1-p) = 2p
\;\Rightarrow\; 8 = 10p
\;\Rightarrow\; \hat{p} = 0.8
$$

Check it's a maximum: the second derivative $-\frac{8}{p^2} - \frac{2}{(1-p)^2}$ is negative everywhere in $(0,1)$ — the log-likelihood is concave, so the critical point is the peak.

### Example 3: other datasets, same formula

3 heads in 10 flips → $\hat p = 0.3$; 5 in 10 → $0.5$; 10 in 10 → $\hat p = 1$ — the MLE says "this coin *always* lands heads", a hint of overfitting on small data that the MAP notebook will address.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

### The three-coin contest

In [ ]:
heads, tails = 8, 2

for name, p in [("coin 1 (p=0.7)", 0.7), ("coin 2 (p=0.5)", 0.5),
                ("coin 3 (p=0.3)", 0.3)]:
    L = p**heads * (1 - p)**tails
    print(f"P(8H 2T | {name}) = {L:.6f}")
print("\ncoin 1 wins: it maximizes P(data | coin).")

### The likelihood as a function of p — and its maximum

In [ ]:
p_grid = np.linspace(0.001, 0.999, 999)
L = p_grid**8 * (1 - p_grid)**2
logL = 8 * np.log(p_grid) + 2 * np.log(1 - p_grid)

print(f"argmax of L(p):     p = {p_grid[np.argmax(L)]:.3f}")
print(f"argmax of log L(p): p = {p_grid[np.argmax(logL)]:.3f}")
print("Same maximizer -- the log changes the heights, never the location of the peak.")

### The derivation with sympy, exactly as on the slide

In [ ]:
p = sp.symbols("p", positive=True)
k, n = sp.symbols("k n", positive=True)

# the 8H2T case
logL_8 = 8 * sp.log(p) + 2 * sp.log(1 - p)
dlogL_8 = sp.diff(logL_8, p)
p_hat_8 = sp.solve(sp.Eq(dlogL_8, 0), p)[0]
print(f"d/dp log L = {dlogL_8}")
print(f"p_hat = {p_hat_8} = {float(p_hat_8)}")

# the general case: k heads in n flips
logL_gen = k * sp.log(p) + (n - k) * sp.log(1 - p)
p_hat_gen = sp.solve(sp.Eq(sp.diff(logL_gen, p), 0), p)[0]
print(f"\ngeneral case: p_hat = {p_hat_gen}")

second = sp.simplify(sp.diff(logL_gen, p, 2))
print(f"second derivative: {second}  (negative on (0,1): a true maximum)")

### $\hat p = k/n$ meets the data

Flip simulated coins and confirm the MLE is always the observed proportion — and that it homes in on the true $p$ as $n$ grows (the LLN at work on the estimator).

In [ ]:
rng = np.random.default_rng(0)

true_p = 0.62
for n_flips in [10, 100, 10_000]:
    flips = rng.random(n_flips) < true_p
    k_obs = flips.sum()
    # numeric maximization on a grid, for the skeptic
    grid = np.linspace(0.001, 0.999, 2001)
    ll = k_obs * np.log(grid) + (n_flips - k_obs) * np.log(1 - grid)
    print(f"n = {n_flips:>6}: k = {k_obs:>5}, k/n = {k_obs/n_flips:.4f}, "
          f"grid argmax = {grid[np.argmax(ll)]:.4f}")
print(f"\ntrue p = {true_p}: the MLE = sample proportion, converging with n.")

## 5. Visualizations

### Likelihood and log-likelihood, side by side

The three candidate coins as dots, the whole curve $L(p) = p^8(1-p)^2$, and its peak at 0.8 — identical argmax in both panels.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))

axes[0].plot(p_grid, L, color="tab:blue", linewidth=2)
for cp, color in [(0.7, "tab:green"), (0.5, "tab:orange"), (0.3, "tab:red")]:
    axes[0].plot(cp, cp**8 * (1 - cp)**2, "o", markersize=10, color=color,
                 label=f"coin p = {cp}")
axes[0].axvline(0.8, color="black", linestyle=":", label="$\\hat{p}$ = 0.8")
axes[0].set_title("likelihood  $L(p) = p^8(1-p)^2$")
axes[0].set_ylabel("$L(p)$")
axes[0].legend(fontsize=9)

axes[1].plot(p_grid, logL, color="tab:purple", linewidth=2)
axes[1].axvline(0.8, color="black", linestyle=":")
axes[1].set_title("log-likelihood  $8\\log p + 2\\log(1-p)$")
axes[1].set_ylabel("$\\log L(p)$")
axes[1].set_ylim(-25, 0)

for ax in axes:
    ax.set_xlabel("p")
    ax.grid(True, alpha=0.3)
plt.suptitle("Same peak at p = 0.8: maximize whichever is nicer (usually the log)")
plt.tight_layout()
plt.show()

### The mechanics of the maximum: derivative crossing zero

In [ ]:
dlogL = 8 / p_grid - 2 / (1 - p_grid)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(p_grid, dlogL, color="tab:blue", linewidth=2,
        label="$\\frac{d}{dp}\\log L = 8/p - 2/(1-p)$")
ax.axhline(0, color="black", linewidth=1)
ax.axvline(0.8, color="tab:red", linestyle="--", linewidth=2,
           label="zero crossing at $\\hat p = 0.8$")
ax.set_ylim(-60, 60)
ax.set_xlabel("p")
ax.set_ylabel("derivative")
ax.set_title("Setting the derivative to zero finds the peak")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

### The likelihood surface for every dataset

One curve per observed head-count $k$ (out of $n = 10$): each peaks exactly at $k/10$ — the whole family of MLEs in one picture.

In [ ]:
plt.figure(figsize=(10, 5))
for k_obs in range(0, 11, 2):
    ll = p_grid**k_obs * (1 - p_grid)**(10 - k_obs)
    ll = ll / ll.max()
    plt.plot(p_grid, ll, linewidth=2, label=f"k = {k_obs}")
    plt.axvline(k_obs / 10, color="gray", linestyle=":", linewidth=0.8)
plt.title("Normalized likelihoods for k heads in 10 flips: each peaks at k/10")
plt.xlabel("p")
plt.ylabel("$L(p)$ (scaled to max 1)")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=9)
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** You observe 3 heads in 10 flips. Which of the three candidate coins (0.7, 0.5, 0.3) wins the likelihood contest? And what is the unrestricted MLE $\hat p$?

**Exercise 2.** Write down the likelihood and log-likelihood for 5 heads in 5 flips. Where is the maximum, and why should the answer worry you a little?

### Intermediate

**Exercise 3.** Redo the video's derivative computation for general $k$ and $n$ *by hand*, then verify each step with sympy (derivative, solution, second-derivative sign).

**Exercise 4.** The log changes nothing: verify numerically on a fine grid that $\arg\max L(p) = \arg\max \log L(p)$ for ten random $(k, n)$ datasets — and explain in one sentence why any increasing transformation preserves the argmax.

### Advanced

**Exercise 5.** Sequence irrelevance: the video computes the probability of "8 heads and 2 tails" as $p^8(1-p)^2$, the probability of one *specific sequence*. The count 8-of-10 actually has probability $\binom{10}{8}p^8(1-p)^2$. Show that including the binomial coefficient changes the likelihood's *values* but not its *argmax* — why do constants in front never matter for MLE?

**Exercise 6.** The MLE's sampling behavior: for the true coin $p = 0.4$, simulate many datasets of size $n = 10, 100, 1000$ and histogram the MLEs $\hat p = k/n$. Verify mean $\approx p$ (unbiased) and std $\approx \sqrt{p(1-p)/n}$ — the sample-proportion facts of Lesson 5, now re-badged as properties of a maximum-likelihood estimator.

## 7. Solutions

In [ ]:
# Exercise 1: 3 heads in 10 flips
h, t = 3, 7
for name, cp in [("0.7", 0.7), ("0.5", 0.5), ("0.3", 0.3)]:
    print(f"P(3H 7T | p={name}) = {cp**h * (1-cp)**t:.6f}")
print(f"coin p = 0.3 wins; unrestricted MLE: p_hat = 3/10 = {3/10}")

In [ ]:
# Exercise 2: 5 heads in 5 flips
grid = np.linspace(0.001, 0.999, 999)
L5 = grid**5
print(f"Exercise 2: L(p) = p^5, log L = 5 log p -- increasing on (0,1],")
print(f"so the maximum sits at the boundary: p_hat = {grid[np.argmax(L5)]:.3f} -> 1.")
print("Worrying because the MLE claims the coin CANNOT land tails after seeing")
print("only 5 flips -- an overconfident, overfit answer on tiny data. (The MAP")
print("estimator will temper exactly this.)")

In [ ]:
# Exercise 3: general derivation with sympy
p_ = sp.symbols("p", positive=True)
k_, n_ = sp.symbols("k n", positive=True)
logL = k_ * sp.log(p_) + (n_ - k_) * sp.log(1 - p_)
d1 = sp.diff(logL, p_)
sol = sp.solve(sp.Eq(d1, 0), p_)[0]
d2 = sp.simplify(sp.diff(logL, p_, 2))
print(f"Exercise 3: d/dp = {d1}")
print(f"solution: p_hat = {sol}")
print(f"second derivative: {d2} < 0 on (0,1) -> maximum confirmed")

In [ ]:
# Exercise 4: argmax invariance under the log
rng = np.random.default_rng(1)
grid = np.linspace(0.001, 0.999, 4001)
ok = True
for _ in range(10):
    n_e = int(rng.integers(5, 50))
    k_e = int(rng.integers(1, n_e))
    L_e = grid**k_e * (1 - grid)**(n_e - k_e)
    ok &= np.argmax(L_e) == np.argmax(np.log(L_e))
print(f"Exercise 4: argmax L == argmax log L in all 10 datasets: {ok}")
print("An increasing transformation preserves order (a > b => f(a) > f(b)),")
print("so whichever p gave the biggest L still gives the biggest log L.")

In [ ]:
# Exercise 5: the binomial coefficient is a spectator
from math import comb
grid = np.linspace(0.001, 0.999, 4001)
L_seq = grid**8 * (1 - grid)**2
L_count = comb(10, 8) * L_seq
print(f"Exercise 5: max L (sequence) = {L_seq.max():.5f} at p = {grid[np.argmax(L_seq)]:.3f}")
print(f"max L (count)    = {L_count.max():.5f} at p = {grid[np.argmax(L_count)]:.3f}")
print("The coefficient C(10,8) = 45 rescales every value equally: multiplying a")
print("function by a positive constant moves no peaks. MLE only compares")
print("likelihoods ACROSS p, so p-independent factors always drop out.")

In [ ]:
# Exercise 6: sampling behavior of the MLE
rng = np.random.default_rng(2)

true_p = 0.4
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=False)
for ax, n_e in zip(axes, [10, 100, 1000]):
    p_hats = (rng.random((30_000, n_e)) < true_p).mean(axis=1)
    ax.hist(p_hats, bins=40, density=True, color="tab:blue", alpha=0.7)
    ax.axvline(true_p, color="tab:red", linestyle="--", linewidth=2)
    ax.set_title(f"n = {n_e}: mean {p_hats.mean():.3f}, "
                 f"std {p_hats.std():.3f}\n(theory {np.sqrt(true_p*(1-true_p)/n_e):.3f})",
                 fontsize=10)
    ax.set_xlabel("$\\hat{p}$")
    ax.grid(True, alpha=0.3)
plt.suptitle("Exercise 6: the MLE p_hat = k/n is the sample proportion -- unbiased, sqrt(p(1-p)/n) spread")
plt.tight_layout()
plt.show()

## 8. Conclusion

- Among finitely many coins, MLE compares $P(\text{data} \mid \text{coin})$: for 8H2T the $p = 0.7$ coin wins ($0.0051$ vs $0.0010$ vs $0.00003$).
- Among **all** coins, the likelihood $L(p) = p^8(1-p)^2$ is a function of $p$; the **log trick** turns the product into the sum $8\log p + 2\log(1-p)$, sharing the same maximizer but far easier to differentiate.
- Setting $\frac{d}{dp}\log L = \frac{8}{p} - \frac{2}{1-p} = 0$ gives $\hat p = \frac{8}{10}$; in general, for $k$ heads among $n$ Bernoulli flips, $L(p) = p^k(1-p)^{n-k}$ and $\hat{p} = \frac{k}{n}$ — **the sample proportion, derived as the maximum-likelihood answer**.
- Constants in front (like $\binom{n}{k}$) and increasing transformations (like $\log$) never move the argmax — the two facts that make MLE computations clean.
- Warning signs at the edges (10-of-10 heads → $\hat p = 1$) preview why priors/regularization will be welcome.

Next: the same game with a continuous population — MLE for the **Gaussian**, where maximizing the likelihood recovers the sample mean and variance.